# Softmax Regression - Companion Notebook

This notebook keeps the workflow code and short explanatory notes for a multiclass softmax-regression example on the wine-quality dataset. The emphasis is practical: inspect the data, fit the model, read the confusion matrix, and connect the resulting probabilities to the calibration ideas covered elsewhere in the course.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.optimize import minimize_scalar
from scipy.special import softmax as scipy_softmax
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, confusion_matrix, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


This setup cell imports the libraries used throughout the notebook. In addition to the modeling tools, it brings in plotting utilities and the calibration helpers used in the final sections.


In [ ]:
# Set pyplot parameters to make things easier to read
plt.rc("axes", linewidth=1.5, labelsize=14)
plt.rc("xtick", labelsize=14)
plt.rc("ytick", labelsize=14)
plt.rc("xtick.major", size=3, width=1.5)
plt.rc("ytick.major", size=3, width=1.5)


This cell only sets the figure style. Keeping it separate makes the rest of the notebook easier to scan and avoids mixing presentation details with the modeling steps.


## Load and Inspect the Dataset

We start by loading the red wine quality dataset and checking what the raw target looks like before creating a simpler multiclass problem.


In [ ]:
wineData = pd.read_csv("../data/winequality-red.csv")
wineData.head()


This first table is mainly for orientation. It lets you verify which columns are available as predictors and confirm that `quality` is currently expressed as an ordinal score rather than as the three classes used later.


In [ ]:
wineData.quality.unique()  # 3 to 8


This quick check shows the distinct quality labels present in the dataset. The original scores run from 3 to 8, so the notebook will collapse them into a smaller set of categories to create a clean multiclass classification problem.


In [ ]:
plt.hist(
    wineData.quality.values,
    bins=[2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5],
    edgecolor="black",
)
plt.xlabel("wine quality", fontsize=18)
plt.ylabel("# of occurrences", fontsize=18)
plt.show()


This histogram gives a compact view of class balance in the original labels. It is useful because any later confusion-matrix pattern should be interpreted in light of how frequent each quality level is to begin with.


## Define a Three-Class Target

To keep the example simple, the original quality score is mapped into three categories:

- `0 = poor` for quality below 6,
- `1 = good` for quality equal to 6,
- `2 = great` for quality above 6.


In [ ]:
c = []
for q in wineData["quality"].values:
    if q < 6:
        c.append(0)
    elif q > 6:
        c.append(2)
    else:
        c.append(1)

wineData["category"] = c
wineData.head()


This relabeling step is what turns the original ordinal score into the target used by the softmax model. It is worth checking the resulting table so you can see the original `quality` column and the derived `category` side by side.


## Prepare the Design Matrix

Softmax regression is fit on the eleven numerical chemistry features. They are standardized before training so that the optimization is not dominated by variables with larger raw scales.


In [ ]:
X = wineData[wineData.columns[0:11]].values
y = wineData["category"].values

scaler = StandardScaler()
Xstan = scaler.fit_transform(X)


This cell isolates the predictor matrix and target vector, then applies standardization. For this notebook the purpose is mainly practical: keep the features on comparable scales before fitting multinomial logistic regression.


In [ ]:
dataStan = pd.DataFrame(data=Xstan, columns=wineData.columns[0:11])
dataStan["category"] = y
dataStan.head()


This standardized table makes the preprocessing result explicit. The numerical columns are now centered and scaled, while the class label remains unchanged. It is useful to inspect this once before fitting the model.


## Fit Softmax Regression

`LogisticRegression` with `multi_class="multinomial"` and an appropriate solver implements softmax regression directly in scikit-learn.


In [ ]:
softReg = LogisticRegression(multi_class="multinomial", solver="lbfgs")
softReg.fit(Xstan, y)


This fit call estimates one parameter vector and one intercept for each class simultaneously under the multinomial objective. In other words, the model is not fitting three separate binary classifiers here; it is fitting one multiclass model with shared normalization through the softmax function.


In [ ]:
softReg.intercept_, softReg.coef_


Because there are three classes, the learned parameters consist of three intercepts and three coefficient vectors. You can read this output mainly as a shape check: one row per class, one coefficient per feature.


In [ ]:
yhat = softReg.predict(Xstan)
dataStan["predict"] = yhat
dataStan.head()


This cell appends the predicted class to the working table. Since the model is being evaluated on the same data it was trained on, the goal at this stage is not honest generalization assessment; it is simply to create the predictions needed for the confusion-matrix walkthrough below.


## Confusion Matrix

The confusion matrix is the most direct way to see which classes the model tends to mix up.


In [ ]:
C = confusion_matrix(dataStan["category"].values, yhat)
confusionMatrix = pd.DataFrame(
    data=C,
    index=["poor(0), true", "good(1), true", "great(2), true"],
    columns=["poor(0), predicted", "good(1), predicted", "great(2), predicted"],
)
confusionMatrix.loc["sum"] = confusionMatrix.sum()
confusionMatrix["sum"] = confusionMatrix.sum(axis=1)
confusionMatrix


The table shows both the raw confusion counts and the marginal totals. This is useful because a large number in one off-diagonal cell may reflect either a genuine systematic confusion or simply the fact that the corresponding true class is very common.


In [ ]:
confMx = confusionMatrix.values[0:3, 0:3]
plt.matshow(confMx, cmap=plt.cm.gray)
plt.show()


This first heatmap shows absolute counts. The diagonal corresponds to correct classifications, so a strong diagonal is the visual signature of a model that is doing the right thing most of the time. Still, absolute counts alone can be misleading when class sizes differ.


In [ ]:
rowSums = confMx.sum(axis=1, keepdims=True)
confMxNorm = confMx / rowSums


Normalizing by row converts each row into conditional proportions given the true class. This answers a more informative question: among wines that truly belong to class `k`, how are they distributed across predicted classes?


In [ ]:
np.fill_diagonal(confMxNorm, 0)
plt.matshow(confMxNorm, cmap=plt.cm.gray)
plt.show()


Zeroing the diagonal removes the correctly classified mass so that only the error structure remains visible. This is often the easiest way to identify which specific misclassification pattern dominates the model’s failures.


## Bridge to Calibration

Softmax outputs are numerically valid probabilities: each row is non-negative and sums to one. But that is not the same as being *well calibrated*. The next sections revisit the same model from a calibration perspective on a proper held-out test split.


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    Xstan, y, test_size=0.30, random_state=42, stratify=y
)

softReg_held = LogisticRegression(multi_class="multinomial", solver="lbfgs", max_iter=500)
softReg_held.fit(X_tr, y_tr)

proba_te = softReg_held.predict_proba(X_te)

class_labels = {0: "poor", 1: "good", 2: "great"}
colors = ["tab:blue", "tab:orange", "tab:green"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for k, (label, color, ax) in enumerate(zip(class_labels.values(), colors, axes)):
    y_binary = (y_te == k).astype(int)
    p_k = proba_te[:, k]

    prob_true, prob_pred = calibration_curve(
        y_binary, p_k, n_bins=8, strategy="uniform"
    )

    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    ax.plot(prob_pred, prob_true, marker="o", color=color, label=f"Class {k} ({label})")
    ax.set_title(f"Class {k}: {label} (one-vs-rest)")
    ax.set_xlabel("Mean Predicted Probability  conf$(B_m)$")
    ax.set_ylabel("Observed Fraction of Positives  acc$(B_m)$")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle("Calibration Curves - Softmax Regression (held-out test set)", fontsize=12)
plt.tight_layout()
plt.show()

print("\nObservation: points are close to the diagonal for all three classes,")
print("confirming that softmax regression trained with cross-entropy")
print("often produces fairly well-calibrated probabilities on in-distribution data.")


This cell redoes the evaluation correctly: the model is trained on one subset and its probabilities are inspected on unseen test data. The calibration curves are plotted one class at a time in a one-vs-rest view, which is the simplest way to adapt binary calibration tools to a multiclass setting.

The main thing to look for is whether each class-specific curve stays close to the diagonal. If so, the model’s reported confidence for that class is reasonably aligned with empirical frequency on held-out data.


In [ ]:
def compute_ece(y_true, y_prob, n_bins=8):
    """Expected Calibration Error with equal-width bins."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n = len(y_true)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.clip(np.digitize(y_prob, bin_edges[:-1], right=False) - 1, 0, n_bins - 1)

    ece = 0.0
    for m in range(n_bins):
        mask = bin_ids == m
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / n) * abs(y_true[mask].mean() - y_prob[mask].mean())
    return ece


print(f"{'Class':<10} {'ECE':>8} {'Brier Score':>14}")
print("-" * 36)
for k, label in class_labels.items():
    y_bin = (y_te == k).astype(int)
    p_k = proba_te[:, k]
    ece_v = compute_ece(y_bin, p_k)
    bs_v = brier_score_loss(y_bin, p_k)
    print(f"{k} ({label}): {ece_v:>8.4f} {bs_v:>14.4f}")


This table complements the curves with scalar summaries. Each row treats one class as a binary one-vs-rest problem, then reports ECE and Brier score for that slice.

Read this output together with the plots rather than in isolation. The metrics tell you *how much* mismatch exists overall for a class, while the reliability diagrams show *where* in the probability range that mismatch occurs.


## Temperature Scaling for the Multiclass Case

Softmax regression is often already reasonably calibrated, but temperature scaling is still a useful reference method because it generalizes naturally to multiclass logits.


In [ ]:
def softmax_temp(logits, T):
    """Apply temperature scaling to a logit matrix and return probabilities."""
    return scipy_softmax(logits / T, axis=1)


def nll_multiclass(T, logits, y_true):
    """Multiclass NLL as a function of temperature T."""
    probs = softmax_temp(logits, T)
    probs = np.clip(probs, 1e-9, 1 - 1e-9)
    n = len(y_true)
    return -np.mean(np.log(probs[np.arange(n), y_true]))


X_tr2, X_cal2, y_tr2, y_cal2 = train_test_split(
    X_tr, y_tr, test_size=0.15, random_state=42, stratify=y_tr
)

softReg_ts = LogisticRegression(multi_class="multinomial", solver="lbfgs", max_iter=500)
softReg_ts.fit(X_tr2, y_tr2)

logits_cal2 = softReg_ts.decision_function(X_cal2)
logits_te2 = softReg_ts.decision_function(X_te)

result = minimize_scalar(
    nll_multiclass,
    args=(logits_cal2, y_cal2),
    bounds=(0.01, 10.0),
    method="bounded",
)
T_opt = result.x

print(f"Optimal temperature T = {T_opt:.4f}")
print("  T approximately 1.0 suggests the model was already close to calibrated.")
print("  T > 1.0 indicates overconfidence and logit shrinkage.")
print("  T < 1.0 indicates underconfidence and logit sharpening.")

proba_uncal_ts = softmax_temp(logits_te2, T=1.0)
proba_cal_ts = softmax_temp(logits_te2, T=T_opt)

nll_before = nll_multiclass(1.0, logits_te2, y_te)
nll_after = nll_multiclass(T_opt, logits_te2, y_te)
print(f"\nNLL on test set - before T-scaling: {nll_before:.4f}")
print(f"NLL on test set - after  T-scaling: {nll_after:.4f}")


This cell fits a single temperature parameter on a dedicated calibration subset. The logic mirrors the binary case, but it operates on the full multiclass logit matrix rather than on one probability column at a time.

The most important number to inspect is `T_opt`. When it stays close to 1, that suggests the original softmax probabilities did not need much correction. The before/after NLL values show whether the calibrated distribution actually improved on held-out data.


In [ ]:
k_demo = 1
y_bin_demo = (y_te == k_demo).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, proba, label in [
    (axes[0], proba_uncal_ts, "Before T-scaling (T=1.00)"),
    (axes[1], proba_cal_ts, f"After T-scaling (T={T_opt:.2f})"),
]:
    pt, pp = calibration_curve(y_bin_demo, proba[:, k_demo], n_bins=8, strategy="uniform")
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    ax.plot(pp, pt, marker="o", color="tab:orange", label=f"Class {k_demo} (good)")
    ax.set_title(label)
    ax.set_xlabel("Mean Predicted Probability  conf$(B_m)$")
    ax.set_ylabel("Observed Fraction of Positives  acc$(B_m)$")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle("Temperature Scaling - Multiclass (class 1: good)", fontsize=12)
plt.tight_layout()
plt.show()


This final comparison uses the `good` class as a representative one-vs-rest slice. The purpose is not to prove that every class behaves identically, but to make the visual effect of temperature scaling easy to inspect.

Read the two panels together with `T_opt` and the NLL values above. If the model was already well calibrated, the curves should look similar and the calibrated version should only make a small correction.
